# CudaSIFT + RootSIFT (GPU) ORB-SLAM3 fork -- KITTI benchmark (Kaggle GPU)

Runs `orbslam3_sift_kitti_ate` (the SIFT-feature fork of ORB-SLAM3, now GPU-accelerated at the **extractor** via [Celebrandil/CudaSift](https://github.com/Celebrandil/CudaSift) + RootSIFT descriptor normalization) against a KITTI odometry sequence. This fork's earlier LightGlue **matcher** experiment (parts 56-58) was reverted after `[reconstruct-diag]` logging showed it producing ~95-97% geometrically-inconsistent matches on KITTI, and the LightGlue paper itself confirmed it was only ever trained/evaluated on wide-baseline data (MegaDepth tourist photos) -- see `DEBUGGING.md`. GPU-accelerating extraction instead sidesteps that negative result entirely.

**Before running:**
1. Notebook settings -> Accelerator -> GPU (T4 x2 or P100). Internet -> On.
2. Add Data -> attach a KITTI odometry (grayscale, sequence 00) dataset, or upload one -- needs `.../image_0/*.png` and a `poses/00.txt`-style ground-truth file somewhere under `/kaggle/input`.
3. Set `REPO_URL` below to this project's GitHub URL (must be pushed first -- this notebook only clones over HTTPS, so the repo needs to be public, or you'll need to bake a token into the URL yourself).

**IMPORTANT -- run `cudasift_probe` before trusting any tracking result**: CudaSift's keypoint scale/octave encoding is NOT the same as OpenCV's `cv::SIFT`, and `ORBextractor.cc`'s conversion is currently a best-effort placeholder pending real measurement (see `analyze/cudasift_probe.cpp`'s doc comment, and Session 14's Stage 0 precedent for why this matters). The setup cell below builds `cudasift_probe` automatically and prints the exact command to run it -- do that FIRST.

The heavy one-time setup (installing apt packages, building g2o from ORB-SLAM3's own `Thirdparty/g2o`, cloning CudaSift, downloading the ONNX Runtime GPU release -- still needed to compile `LightGlueMatcher.cc`, which stays in the tree unused) is cached under `kaggle/g2o_build`, `kaggle/cudasift_src`, and `kaggle/onnxruntime` for the rest of the session -- rerunning the run cell after tweaking `START_FRAME`/`MAX_FRAMES` does not repeat it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")

In [ ]:
# One-time setup: apt deps, g2o build, CudaSift clone, ONNX Runtime GPU
# download, cmake configure + build (USE_CUDASIFT=1 by default). Safe to
# rerun (skips steps whose output already exists). Also builds
# cudasift_probe and prints the command to run it -- do that before
# trusting the benchmark run below.
!cd {REPO_DIR} && bash kaggle/setup_and_run.sh

The cell above both builds AND runs a default 0-1000-frame benchmark (env var defaults in `kaggle/setup_and_run.sh`). To rerun with different frame ranges without repeating the build, use the cell below instead.

In [ ]:
import os

env = os.environ.copy()
env["SKIP_BUILD"] = "1"
env["USE_CUDASIFT"] = "1"  # set to "0" to compare against the plain CPU cv::SIFT path
env["START_FRAME"] = "0"
env["MAX_FRAMES"] = "1000"
env["OUT_PREFIX"] = "/kaggle/working/cudasift_1_1000"
# Uncomment and set explicitly if auto-detection under /kaggle/input picks the wrong dataset:
# env["KITTI_SEQ_DIR"] = "/kaggle/input/<your-dataset>/sequences/00"
# env["KITTI_POSES"] = "/kaggle/input/<your-dataset>/poses/00.txt"

import subprocess, sys
result = subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env)
sys.exit(result.returncode) if result.returncode != 0 else None

In [ ]:
# Fragment-by-fragment ATE report and the [sbp-diag] match-quality summary
# are printed directly to stdout by the run cell above (search for
# "[atlas-coverage]" / "[fragment" / "[sbp-diag]"). The keyframe trajectory
# itself is written to OUT_PREFIX + "_KeyFrameTrajectory.txt":
!tail -n 20 /kaggle/working/cudasift_1_1000_KeyFrameTrajectory.txt

## Auto-run whole-KITTI subset (00/01/02/05/08) + results table

Runs the full recipe (SQPnP + init-fix + epipolar bridge, commit `ca00d94`) on each
sequence with `SKIP_BUILD=1` (build once via Cell 3 first), captures each run's stdout,
and prints the results table. Baseline (pre-`ca00d94`) is a separate run — see the note
at the end. Adjust `SEQS` / `SEQ_BASE` if your dataset path differs.

In [ ]:
import os, re, glob, subprocess

SEQS = ["00", "01", "02", "05", "08"]

# Auto-locate the KITTI sequences dir under /kaggle/input (…/sequences/<NN>/image_0)
cands = sorted(glob.glob("/kaggle/input/**/sequences", recursive=True))
SEQ_BASE = cands[0] if cands else "/kaggle/input/kitti-odometry-gray/dataset/sequences"
print("SEQ_BASE =", SEQ_BASE, "  (override manually if wrong)")

rows = []
for seq in SEQS:
    seq_dir = f"{SEQ_BASE}/{seq}"
    poses   = f"{REPO_DIR}/kitti_poses/{seq}.txt"
    if not os.path.isdir(f"{seq_dir}/image_0"):
        print(f"[skip {seq}] no image_0 at {seq_dir}"); continue
    if not os.path.isfile(poses):
        print(f"[skip {seq}] no poses at {poses}"); continue
    logf = f"/kaggle/working/seq{seq}_full.log"
    env = os.environ.copy()
    env.update(SKIP_BUILD="1", USE_CUDASIFT="1", START_FRAME="0", MAX_FRAMES="0",
               KITTI_SEQ_DIR=seq_dir, KITTI_POSES=poses,
               OUT_PREFIX=f"/kaggle/working/seq{seq}_full")
    print(f"=== running seq{seq} (full) … ===", flush=True)
    with open(logf, "w") as f:
        subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env,
                       stdout=f, stderr=subprocess.STDOUT)
    txt = open(logf, errors="ignore").read()
    frags = re.findall(r"pathLen=([\d.]+)m ATE\(rmse[^=]*=([\d.]+)/", txt)
    cov_m = sum(float(p) for p, _ in frags)
    m = re.search(r"GT path length:\s*([\d.]+)", txt)
    gt_m = float(m.group(1)) if m else 0.0
    worst = max((float(a) for _, a in frags), default=0.0)
    resets = len(re.findall(r"A new map is started|Reseting current map", txt))
    bridge = len(re.findall(r"\[bridge\].*SUCCESS", txt))
    ninit  = len(re.findall(r"mono-init.*SUCCESS", txt))
    nfail  = len(re.findall(r"reconstruct FAILED", txt))
    initp  = 100.0 * ninit / max(1, ninit + nfail)
    covp   = 100.0 * cov_m / gt_m if gt_m else 0.0
    rows.append((seq, cov_m, covp, worst, resets, bridge, initp, len(frags)))
    print(f"    seq{seq}: cov={cov_m:.1f}m/{covp:.1f}%  worstATE={worst:.2f}m  "
          f"resets={resets}  bridge={bridge}  init%={initp:.1f}  frags={len(frags)}", flush=True)

print("\n\n=== RESULTS TABLE (SQPnP + init-fix + bridge, ca00d94, CudaSIFT 5000) ===\n")
print("| Seq | Cov(m) | Cov% | worstATE | Resets | Bridge | Init% | Frags |")
print("|-----|--------|------|----------|--------|--------|-------|-------|")
for seq, cm, cp, wa, rs, br, ip, nf in rows:
    print(f"| {seq} | {cm:.1f} | {cp:.1f}% | {wa:.2f}m | {rs} | {br} | {ip:.1f}% | {nf} |")

print("\nBaseline (pre-ca00d94, KLT+reset) for comparison: run once with")
print("  !cd {REPO_DIR} && git stash && git checkout ca00d94^ && <rebuild via Cell 3, SKIP_BUILD unset>")
print("then re-run this cell, then `git checkout main` (or git stash pop) to restore.")

## Auto-run 00–05 x toggle sweep + results table (v2)

Sweeps a small set of `TRACKER`/`BRIDGE`/`INITFIX` configs across **sequences 00–05**
using `SKIP_BUILD=1` (build once via Cell 3 first) -- no rebuild needed between configs
since they're all runtime env-var toggles (commit adding TRACKER/BRIDGE/INITFIX).
Edit `SEQS` (e.g. `["00"]` to test one sequence only) or `CONFIGS` to narrow the sweep.

In [ ]:
import os, re, glob, subprocess

SEQS = [f"{i:02d}" for i in range(0, 6)]   # 00..05 -- edit to e.g. ["00"] for a single seq

# name -> (TRACKER, BRIDGE, INITFIX)
CONFIGS = {
    "baseline_klt_nobridge":  ("klt",   "off",  "off"),
    "klt_bridge_both":        ("klt",   "both", "on"),
    "sqpnp_nobridge":         ("sqpnp", "off",  "on"),
    "sqpnp_bridge_rl":        ("sqpnp", "rl",   "on"),
    "sqpnp_bridge_lost":      ("sqpnp", "lost", "on"),
    "sqpnp_bridge_both":      ("sqpnp", "both", "on"),
}

cands = sorted(glob.glob("/kaggle/input/**/sequences", recursive=True))
SEQ_BASE = cands[0] if cands else "/kaggle/input/kitti-odometry-gray/dataset/sequences"
print("SEQ_BASE =", SEQ_BASE, "  (override manually if wrong)")

rows = []
for cfg_name, (tracker, bridge, initfix) in CONFIGS.items():
    for seq in SEQS:
        seq_dir = f"{SEQ_BASE}/{seq}"
        poses   = f"{REPO_DIR}/kitti_poses/{seq}.txt"
        if not os.path.isdir(f"{seq_dir}/image_0") or not os.path.isfile(poses):
            print(f"[skip {cfg_name}/{seq}] missing images or poses"); continue
        logf = f"/kaggle/working/{cfg_name}_seq{seq}.log"
        env = os.environ.copy()
        env.update(SKIP_BUILD="1", USE_CUDASIFT="1", START_FRAME="0", MAX_FRAMES="0",
                   KITTI_SEQ_DIR=seq_dir, KITTI_POSES=poses,
                   OUT_PREFIX=f"/kaggle/working/{cfg_name}_seq{seq}",
                   TRACKER=tracker, BRIDGE=bridge, INITFIX=initfix)
        print(f"=== {cfg_name} / seq{seq} (TRACKER={tracker} BRIDGE={bridge} INITFIX={initfix}) ===", flush=True)
        with open(logf, "w") as f:
            subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env,
                           stdout=f, stderr=subprocess.STDOUT)
        txt = open(logf, errors="ignore").read()
        frags = re.findall(r"pathLen=([\d.]+)m ATE\(rmse[^=]*=([\d.]+)/", txt)
        cov_m = sum(float(p) for p, _ in frags)
        m = re.search(r"GT path length:\s*([\d.]+)", txt)
        gt_m = float(m.group(1)) if m else 0.0
        worst = max((float(a) for _, a in frags), default=0.0)
        resets = len(re.findall(r"A new map is started|Reseting current map", txt))
        bridge_ok = len(re.findall(r"\[bridge\].*SUCCESS", txt))
        ninit  = len(re.findall(r"mono-init.*SUCCESS", txt))
        nfail  = len(re.findall(r"reconstruct FAILED", txt))
        initp  = 100.0 * ninit / max(1, ninit + nfail)
        covp   = 100.0 * cov_m / gt_m if gt_m else 0.0
        rows.append((cfg_name, seq, cov_m, covp, worst, resets, bridge_ok, initp, len(frags)))
        print(f"    cov={cov_m:.1f}m/{covp:.1f}%  worstATE={worst:.2f}m  resets={resets}  "
              f"bridgeOK={bridge_ok}  init%={initp:.1f}  frags={len(frags)}", flush=True)

print("\n\n=== RESULTS TABLE (toggle sweep, 00-05) ===\n")
print("| Config | Seq | Cov(m) | Cov% | worstATE | Resets | BridgeOK | Init% | Frags |")
print("|--------|-----|--------|------|----------|--------|----------|-------|-------|")
for cfg, seq, cm, cp, wa, rs, br, ip, nf in rows:
    print(f"| {cfg} | {seq} | {cm:.1f} | {cp:.1f}% | {wa:.2f}m | {rs} | {br} | {ip:.1f}% | {nf} |")